# 🎯 PaliGemma Math Recognition Training (A100/H100)

**Streamlined training pipeline for Google Colab Pro**

This notebook will:
1. Mount Google Drive for persistent storage
2. Install dependencies
3. Use existing dataset (fast extraction to local SSD)
4. Train PaliGemma-3B with LoRA (~15-20 hours on A100)
5. Auto-save checkpoints to Drive

**Setup Requirements:**
- Google Colab Pro (for A100/H100 access)
- HuggingFace token with PaliGemma access
- Dataset tarball in Google Drive

In [ ]:
# ============================================================================
# 1️⃣ SETUP: Mount Drive & Configure Environment
# ============================================================================

import os
from google.colab import drive, userdata

# Mount Google Drive for persistent storage
drive.mount('/content/drive')

# Create working directory
WORK_DIR = '/content/drive/MyDrive/math-training'
os.makedirs(WORK_DIR, exist_ok=True)

print(f"✅ Working directory: {WORK_DIR}")
print("✅ Checkpoints will be saved to Google Drive")

In [ ]:
# Check GPU allocation
!nvidia-smi

import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"\n🖥️  Device: {gpu_name}")
print(f"📊 VRAM: {vram_gb:.1f} GB")

# Auto-detect batch size based on GPU
if 'H100' in gpu_name:
    BATCH_SIZE = 16
    print("✅ H100 detected - using batch size 16")
elif 'A100' in gpu_name:
    BATCH_SIZE = 12
    print("✅ A100 detected - using batch size 12 (safer for 40GB)")
else:
    BATCH_SIZE = 8
    print(f"⚠️  Using {gpu_name} - batch size 8 (may be slow)")

In [ ]:
# Install dependencies (fix version compatibility)
!pip install -q torch>=2.2.0 transformers>=4.50.0 \
    datasets accelerate sentencepiece protobuf

# Install compatible versions to avoid import errors
!pip install -q --upgrade huggingface_hub>=0.20.0
!pip install -q peft>=0.13.0
!pip install -q bitsandbytes>=0.45.0
!pip install -q matplotlib pillow numpy tqdm timm

print("✅ Dependencies installed")

In [ ]:
# ============================================================================
# 2️⃣ AUTHENTICATION: HuggingFace Token
# ============================================================================

from huggingface_hub import login

# Get token from Colab secrets
# Go to 🔑 (left sidebar) → Add secret → Name: HF_TOKEN → Value: your_token
# Create token at: https://huggingface.co/settings/tokens
# Request access to: https://huggingface.co/google/paligemma-3b-pt-224

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("✅ HuggingFace authenticated")
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("\nPlease:")
    print("1. Create token: https://huggingface.co/settings/tokens")
    print("2. Request access: https://huggingface.co/google/paligemma-3b-pt-224")
    print("3. Add to Colab secrets (🔑 icon): Name=HF_TOKEN")
    raise

In [ ]:
# ============================================================================
# 3️⃣ DATASET: Fast extraction to local SSD
# ============================================================================

import subprocess
import glob
import time

DATASET_URL = "https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz"

# Use LOCAL /content for fast I/O (not Drive)
LOCAL_DATA_DIR = "/content/mathwriting-2024"
DRIVE_TARBALL = f"{WORK_DIR}/mathwriting-2024.tgz"
DRIVE_DATA_DIR = f"{WORK_DIR}/mathwriting-2024"

print("🔍 Checking for dataset...")
dataset_ready = False

# Check if already extracted locally (from previous session)
if os.path.exists(f"{LOCAL_DATA_DIR}/train") and \
   os.path.exists(f"{LOCAL_DATA_DIR}/valid") and \
   os.path.exists(f"{LOCAL_DATA_DIR}/test"):
    train_count = len(glob.glob(f"{LOCAL_DATA_DIR}/train/*.inkml"))
    if train_count > 100000:  # Verify it's complete
        print(f"✅ Dataset already in local storage (fast!)")
        dataset_ready = True
        for split in ['train', 'valid', 'test']:
            count = len(glob.glob(f"{LOCAL_DATA_DIR}/{split}/*.inkml"))
            print(f"   {split}: {count:,} files")

# If not in local storage, check for tarball and extract
if not dataset_ready:
    # Check for tarball in Drive
    if os.path.exists(DRIVE_TARBALL):
        print(f"✅ Found tarball in Drive: {DRIVE_TARBALL}")
        tarball_path = DRIVE_TARBALL
    elif os.path.exists(f"{WORK_DIR}/mathwriting-2024.tgz"):
        print(f"✅ Found tarball in Drive")
        tarball_path = f"{WORK_DIR}/mathwriting-2024.tgz"
    elif os.path.exists("/content/drive/MyDrive/mathwriting-2024.tgz"):
        print(f"✅ Found tarball in Drive root")
        tarball_path = "/content/drive/MyDrive/mathwriting-2024.tgz"
    else:
        # Download if not found
        print(f"📥 Downloading dataset to Drive (~2.9 GB)...")
        !wget -q --show-progress {DATASET_URL} -O {DRIVE_TARBALL}
        tarball_path = DRIVE_TARBALL
    
    # Extract to LOCAL /content (FAST!)
    print(f"\n📦 Extracting to local SSD (fast storage)...")
    print(f"   From: {tarball_path}")
    print(f"   To: /content/ (local SSD)")
    print(f"   This takes ~30 seconds (vs 20+ minutes on Drive)\n")
    
    start = time.time()
    result = subprocess.run(
        ["tar", "-xzf", tarball_path, "-C", "/content/"],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        elapsed = time.time() - start
        print(f"✅ Extraction complete in {elapsed:.1f} seconds!")
        
        # Verify
        for split in ['train', 'valid', 'test']:
            count = len(glob.glob(f"{LOCAL_DATA_DIR}/{split}/*.inkml"))
            print(f"   {split}: {count:,} files")
        dataset_ready = True
    else:
        print(f"❌ Extraction failed: {result.stderr}")
        raise RuntimeError("Dataset extraction failed")

# Use local data directory for training
DATA_DIR = LOCAL_DATA_DIR
print(f"\n✅ Using dataset at: {DATA_DIR} (local SSD - fast I/O!)")

In [ ]:
# ============================================================================
# 4️⃣ PROJECT FILES: Clone from GitHub
# ============================================================================

os.chdir('/content')  # Work from /content (fast local storage)

# Clone repo if needed
if not os.path.exists('data_preprocessing.py'):
    print("📥 Cloning project files...")
    !git clone https://github.com/hudsonmp/realtime-math.git temp_repo
    !cp temp_repo/*.py .
    !rm -rf temp_repo
    print("✅ Project files ready")
else:
    print("✅ Project files already present")

# Verify
required = ['data_preprocessing.py', 'train.py']
for f in required:
    assert os.path.exists(f), f"Missing {f}"
    print(f"   ✅ {f}")

In [ ]:
# ============================================================================
# 5️⃣ TRAINING: PaliGemma-3B + LoRA (A100/H100 Optimized)
# ============================================================================

import torch
from torch.utils.data import DataLoader
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, get_scheduler
from peft import LoraConfig, get_peft_model
from data_preprocessing import MathWritingDataset, LaTeXTokenizer
from tqdm import tqdm
import time

# ============================================================================
# HYPERPARAMETERS (Optimized for A100/H100)
# ============================================================================

EPOCHS = 10
# BATCH_SIZE set above based on GPU detection
GRAD_ACCUM = 3 if BATCH_SIZE == 12 else 2  # Keep effective batch ~36
LEARNING_RATE = 2e-4
WARMUP_STEPS = 500
MAX_GRAD_NORM = 1.0

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Paths
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = "cuda"

print("="*70)
print("🚀 TRAINING CONFIGURATION")
print("="*70)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"LoRA Rank: {LORA_R}")
print(f"Checkpoints: {CHECKPOINT_DIR} (Drive - persistent)")
print(f"Dataset: {DATA_DIR} (local SSD - fast!)")
print("="*70)

# ============================================================================
# LOAD MODEL
# ============================================================================

print("\n📦 Loading PaliGemma-3B...")
processor = AutoProcessor.from_pretrained("google/paligemma-3b-pt-224")

model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    torch_dtype=torch.bfloat16,
    device_map=None
)

# Add LoRA adapters
print("🔧 Applying LoRA...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.to(device)

model.print_trainable_parameters()

# ============================================================================
# LOAD DATASETS
# ============================================================================

print("\n📊 Loading datasets...")
train_ds = MathWritingDataset(DATA_DIR, split='train')
valid_ds = MathWritingDataset(DATA_DIR, split='valid')

print(f"   Train: {len(train_ds):,} samples")
print(f"   Valid: {len(valid_ds):,} samples")

latex_tokenizer = LaTeXTokenizer()

def collate_fn(batch):
    """Collate function for DataLoader."""
    stroke_texts = [item['stroke_text'] for item in batch]
    images = [item['image'] for item in batch]
    labels = [item['label'] for item in batch]

    inputs = processor(
        text=stroke_texts,
        images=images,
        padding="longest",
        truncation=True,
        max_length=1024,
        return_tensors="pt"
    )

    label_encodings = processor.tokenizer(
        labels,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    batch_size = inputs['input_ids'].shape[0]
    seq_length = inputs['input_ids'].shape[1]
    labels_tensor = torch.full((batch_size, seq_length), -100, dtype=torch.long)

    for i, label_ids in enumerate(label_encodings['input_ids']):
        label_length = (label_ids != processor.tokenizer.pad_token_id).sum().item()
        labels_tensor[i, -label_length:] = label_ids[:label_length]

    inputs['labels'] = labels_tensor
    return inputs

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# ============================================================================
# TRAINING SETUP
# ============================================================================

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

num_training_steps = EPOCHS * len(train_loader) // GRAD_ACCUM
scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=num_training_steps
)

print(f"\n📈 Training Setup:")
print(f"   Steps per epoch: {len(train_loader)}")
print(f"   Total optimization steps: {num_training_steps}")
print(f"   Warmup steps: {WARMUP_STEPS}")

# ============================================================================
# TRAINING LOOP
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70)

model.train()
global_step = 0
best_cer = float('inf')
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\n{'='*70}")
    print(f"📅 EPOCH {epoch + 1}/{EPOCHS}")
    print('='*70)
    
    epoch_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
        
        epoch_loss += loss.item() * GRAD_ACCUM
        
        progress_bar.set_postfix({
            'loss': f'{loss.item() * GRAD_ACCUM:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}'
        })
    
    avg_loss = epoch_loss / len(train_loader)
    elapsed = (time.time() - start_time) / 3600
    
    print(f"\n📊 Epoch {epoch+1} Results:")
    print(f"   Train Loss: {avg_loss:.4f}")
    print(f"   Time Elapsed: {elapsed:.2f} hours")
    
    # VALIDATION
    print("\n🔍 Running validation...")
    model.eval()
    val_loss = 0
    total_cer = 0
    num_samples = 0
    
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc="Validation"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            labels = batch['labels'].to(device)
            
            outputs = model(**{**inputs, 'labels': labels})
            val_loss += outputs.loss.item()
            
            generated = model.generate(**inputs, max_length=64)
            
            for pred_ids, label_ids in zip(generated, labels):
                pred_text = processor.decode(pred_ids, skip_special_tokens=True)
                label_text = processor.decode(label_ids[label_ids != -100], skip_special_tokens=True)
                cer = latex_tokenizer.compute_cer(pred_text, label_text)
                total_cer += cer
                num_samples += 1
    
    avg_val_loss = val_loss / len(valid_loader)
    avg_cer = total_cer / num_samples if num_samples > 0 else 0
    
    print(f"\n📊 Validation Results:")
    print(f"   Val Loss: {avg_val_loss:.4f}")
    print(f"   CER: {avg_cer:.4f}")
    
    model.train()
    
    # CHECKPOINTING
    if avg_cer < best_cer:
        print(f"\n🎉 New best CER: {best_cer:.4f} → {avg_cer:.4f}")
        best_cer = avg_cer
        save_path = f"{CHECKPOINT_DIR}/best_model"
        model.save_pretrained(save_path)
        processor.save_pretrained(save_path)
        print(f"✅ Best model saved to {save_path}")
    
    if (epoch + 1) % 2 == 0:
        save_path = f"{CHECKPOINT_DIR}/epoch_{epoch+1}"
        model.save_pretrained(save_path)
        print(f"💾 Checkpoint saved: {save_path}")

# FINAL SAVE
final_path = f"{CHECKPOINT_DIR}/final_model"
model.save_pretrained(final_path)
processor.save_pretrained(final_path)

total_time = (time.time() - start_time) / 3600

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)
print(f"Total time: {total_time:.2f} hours")
print(f"Best CER: {best_cer:.4f}")
print(f"\nModels saved to: {CHECKPOINT_DIR}")
print("   - best_model/     (lowest CER)")
print("   - final_model/    (last epoch)")
print("   - epoch_X/        (periodic checkpoints)")
print("\n✅ All files saved to Google Drive (persistent)")
print("="*70)

In [ ]:
# ============================================================================
# 6️⃣ OPTIONAL: Test Inference
# ============================================================================

# Reload modules to avoid import errors
import importlib
import sys

# Force reload peft
if 'peft' in sys.modules:
    importlib.reload(sys.modules['peft'])

from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
import torch

print("Loading best model for inference...")

base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    f"{CHECKPOINT_DIR}/best_model"
)

processor = AutoProcessor.from_pretrained(f"{CHECKPOINT_DIR}/best_model")

print("✅ Model loaded!")

# Test on a sample
from data_preprocessing import MathWritingDataset

test_ds = MathWritingDataset(DATA_DIR, split='test')
sample = test_ds[0]

inputs = processor(
    text=sample['stroke_text'],
    images=sample['image'],
    return_tensors="pt"
).to("cuda")

generated = model.generate(**inputs, max_length=64)
prediction = processor.decode(generated[0], skip_special_tokens=True)

print(f"\n🧪 Sample Test:")
print(f"   Ground Truth: {sample['label']}")
print(f"   Prediction:   {prediction}")

---

## 📝 Notes

**Training Time:**
- A100 (40GB): ~15-20 hours for 10 epochs
- H100 (80GB): ~12-18 hours for 10 epochs

**Storage Strategy:**
- ⚡ Dataset: Local SSD (`/content`) - FAST extraction (~30s) & I/O
- 💾 Checkpoints: Google Drive - Persistent across sessions
- 📦 Tarball: Google Drive - Download once, reuse forever

**Why This Is Fast:**
- Extraction to `/content`: ~30 seconds (vs 20+ min to Drive)
- Training data loading: Fast local SSD I/O
- Checkpoints to Drive: Only small LoRA weights (~500MB)

**Colab Pro Features:**
- ✅ Survives tab closures
- ✅ Checkpoints saved to Google Drive
- ✅ Dataset auto-extracts from Drive tarball on reconnect